# H-03: Cross-Category Performance Comparison

**Question:** Does the agent handle all fault categories equally — or does it struggle with specific ones?

| | |
|---|---|
| **H₀ (Null):** | Performance is the same across all fault categories. |
| **Hₐ (Alternative):** | At least one fault category has significantly different performance. |
| **Certification rule:** | If disparity is significant with medium/large effect, the agent's weakest category is flagged. |
| **Metrics:** | time_to_detect, time_to_mitigate, reasoning_quality_score, hallucination_score |
| **Pipeline:** | Shapiro-Wilk (informational) → Kruskal-Wallis (omnibus) → Mann-Whitney U + A12 (Holm-corrected pairwise) |

### Aggregation Strategy
- **Pooled per-run:** All sub-fault values pooled within each category for statistical tests
- **Within-category heterogeneity:** KW test among sub-faults flags categories where pooling may confound results
- **Equal-weight IQM:** Reported alongside pooled stats for H-01 consistency

### Why always nonparametric?
SRE metric data is rarely normally distributed (bimodal network, right-skewed resource). Kruskal-Wallis + Mann-Whitney are robust regardless of distribution shape.


In [1]:
import sys, os
import json
import numpy as np
from pathlib import Path
from collections import defaultdict

project_root = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(project_root))

data_root = project_root / "hypothesis_framework" / "data" / "input"

def load_runs(category_dir):
    runs = []
    for f in sorted(category_dir.glob("*.json")):
        run = json.loads(f.read_text(encoding="utf-8"))
        runs.append(run)
    return runs

def build_subfault_data(runs, metric_name, detected_only=True, source="quantitative"):
    """Group metric values by sub-fault name."""
    grouped = defaultdict(list)
    for run in runs:
        q = run["quantitative"]
        if detected_only and q.get("fault_detected") != "Yes":
            continue
        fname = run["fault_name"]
        if source == "quantitative":
            val = q.get(metric_name)
        else:
            val = run["qualitative"].get(metric_name)
        if val is not None:
            grouped[fname].append(float(val))
    return dict(grouped)

# Load all runs per category
categories = {}
for cat_name in ["application_fault", "network_fault", "resource_fault"]:
    cat_dir = data_root / cat_name
    if cat_dir.exists():
        categories[cat_name] = load_runs(cat_dir)

print("=== Data Loaded ===")
for cat, runs in categories.items():
    total = len(runs)
    detected = sum(1 for r in runs if r["quantitative"].get("fault_detected") == "Yes")
    faults = set(r["fault_name"] for r in runs)
    print(f"  {cat}: {total} runs, {detected} detected ({detected/total:.0%})")
    for fn in sorted(faults):
        fn_runs = [r for r in runs if r["fault_name"] == fn]
        fn_det = sum(1 for r in fn_runs if r["quantitative"].get("fault_detected") == "Yes")
        print(f"    {fn}: {len(fn_runs)} runs, {fn_det} detected")


=== Data Loaded ===
  application_fault: 60 runs, 51 detected (85%)
    container-kill: 30 runs, 28 detected
    pod-delete: 30 runs, 23 detected
  network_fault: 90 runs, 39 detected (43%)
    pod-dns-error: 30 runs, 13 detected
    pod-network-corruption: 30 runs, 14 detected
    pod-network-loss: 30 runs, 12 detected
  resource_fault: 90 runs, 70 detected (78%)
    disk-fill: 30 runs, 25 detected
    pod-cpu-hog: 30 runs, 25 detected
    pod-memory-hog: 30 runs, 20 detected


---
## Step 1: Per Sub-Fault Distribution Overview

Before comparing categories, let's see how sub-faults behave within each category.
This motivates the within-category heterogeneity check.


In [2]:
from scipy.stats import trim_mean

print("time_to_detect per sub-fault (detected-only):")
print("=" * 85)

for cat, runs in categories.items():
    subfault_data = build_subfault_data(runs, "time_to_detect")
    print(f"\n{cat}:")
    for fname, vals in sorted(subfault_data.items()):
        arr = np.array(vals)
        iqm = trim_mean(arr, 0.25) if len(arr) >= 4 else np.mean(arr)
        print(f"  {fname:25s}: n={len(arr):3d}  IQM={iqm:7.1f}s  "
              f"median={np.median(arr):7.1f}s  std={np.std(arr, ddof=1):6.1f}s")

    # Pooled stats
    all_vals = [v for vals in subfault_data.values() for v in vals]
    pooled_iqm = trim_mean(all_vals, 0.25) if len(all_vals) >= 4 else np.mean(all_vals)
    print(f"  {'POOLED':25s}: n={len(all_vals):3d}  IQM={pooled_iqm:7.1f}s")


time_to_detect per sub-fault (detected-only):

application_fault:
  container-kill           : n= 28  IQM=  122.9s  median=  120.1s  std=  25.1s
  pod-delete               : n= 23  IQM=  131.2s  median=  130.9s  std=  37.1s
  POOLED                   : n= 51  IQM=  125.2s

network_fault:
  pod-dns-error            : n= 13  IQM=   70.0s  median=   60.0s  std= 259.5s
  pod-network-corruption   : n= 14  IQM=  215.4s  median=  136.3s  std= 323.7s
  pod-network-loss         : n= 12  IQM=   86.7s  median=   77.0s  std= 215.4s
  POOLED                   : n= 39  IQM=  120.6s

resource_fault:
  disk-fill                : n= 25  IQM=  223.4s  median=  226.7s  std=  45.1s
  pod-cpu-hog              : n= 25  IQM=  235.5s  median=  262.7s  std=  66.7s
  pod-memory-hog           : n= 20  IQM=  290.7s  median=  289.6s  std=  76.8s
  POOLED                   : n= 70  IQM=  241.9s


---
## Step 2: Run H-03 — time_to_detect

Kruskal-Wallis omnibus test across 3 categories. If significant, pairwise Mann-Whitney U + A12 with Holm-Bonferroni correction.


In [3]:
from hypothesis_framework.scripts.hypothesis_tests.h03_cross_category_comparison import run_cross_category_test

# Build input: {category: {sub_fault: [values]}}
ttd_data = {}
for cat, runs in categories.items():
    ttd_data[cat] = build_subfault_data(runs, "time_to_detect")

h03_ttd = run_cross_category_test(ttd_data, metric_name="time_to_detect")

print(f"H-03: {h03_ttd.hypothesis_name}")
print(f"Metric: {h03_ttd.metric_name}")
print(f"Overall: {h03_ttd.overall_assessment}")
print("=" * 85)

# Per-category details
print("\n--- Per-Category Details ---")
for c in h03_ttd.per_category:
    normal_lbl = "normal" if c.is_normal else "non-normal"
    het_lbl = " ⚠️ HETEROGENEOUS" if c.within_heterogeneous else " (homogeneous)"
    print(f"\n  {c.category} (n={c.n}, {c.n_sub_faults} sub-faults, {normal_lbl}{het_lbl}):")
    print(f"    Pooled:       IQM={c.pooled_iqm:.1f}s  median={c.pooled_median:.1f}s  std={c.pooled_std:.1f}s")
    print(f"    Equal-weight: IQM={c.equal_weight_iqm:.1f}s")
    if c.within_kw_p is not None:
        print(f"    Within-cat KW: p={c.within_kw_p:.4f}")
    for sf in c.sub_faults:
        print(f"      {sf.fault_name:25s}: n={sf.n:3d}  IQM={sf.iqm:7.1f}s  median={sf.median:7.1f}s")

# Omnibus
print(f"\n--- Omnibus: {h03_ttd.test_used} ---")
print(f"  H = {h03_ttd.omnibus_statistic:.4f},  p = {h03_ttd.omnibus_p:.6f}")
print(f"  Significant: {h03_ttd.omnibus_significant}")

# Pairwise
if h03_ttd.pairwise:
    print(f"\n--- Pairwise Post-Hoc ({h03_ttd.correction_method}) ---")
    for pw in h03_ttd.pairwise:
        sig = "✗" if pw.significant else "✓"
        print(f"  {pw.pair:50s}: U={pw.u_statistic:8.1f}  "
              f"p_raw={pw.p_value_raw:.4f}  p_adj={pw.p_value_adjusted:.4f}  "
              f"A12={pw.a12:.3f} ({pw.effect_magnitude})  [{sig}]")
else:
    print("\n  No pairwise tests (omnibus not significant).")

if h03_ttd.warnings:
    print(f"\nWarnings:")
    for w in h03_ttd.warnings:
        print(f"  - {w}")


H-03: Cross-Category Performance Comparison
Metric: time_to_detect
Overall: significant_category_disparity

--- Per-Category Details ---

  application_fault (n=51, 2 sub-faults, normal (homogeneous)):
    Pooled:       IQM=125.2s  median=123.5s  std=31.1s
    Equal-weight: IQM=127.0s
    Within-cat KW: p=0.3586
      container-kill           : n= 28  IQM=  122.9s  median=  120.1s
      pod-delete               : n= 23  IQM=  131.2s  median=  130.9s

  network_fault (n=39, 3 sub-faults, non-normal (homogeneous)):
    Pooled:       IQM=120.6s  median=80.2s  std=271.9s
    Equal-weight: IQM=124.0s
    Within-cat KW: p=0.2013
      pod-dns-error            : n= 13  IQM=   70.0s  median=   60.0s
      pod-network-corruption   : n= 14  IQM=  215.4s  median=  136.3s
      pod-network-loss         : n= 12  IQM=   86.7s  median=   77.0s

  resource_fault (n=70, 3 sub-faults, normal ⚠️ HETEROGENEOUS):
    Pooled:       IQM=241.9s  median=250.4s  std=67.5s
    Equal-weight: IQM=249.9s
    Within

---
## Step 3: Run H-03 — time_to_mitigate


In [4]:
ttm_data = {}
for cat, runs in categories.items():
    ttm_data[cat] = build_subfault_data(runs, "time_to_mitigate")

h03_ttm = run_cross_category_test(ttm_data, metric_name="time_to_mitigate")

print(f"H-03: {h03_ttm.metric_name}  |  {h03_ttm.overall_assessment}")
print("=" * 85)
for c in h03_ttm.per_category:
    het_lbl = " ⚠️" if c.within_heterogeneous else ""
    print(f"  {c.category:20s}: pooled IQM={c.pooled_iqm:7.1f}s  EW IQM={c.equal_weight_iqm:7.1f}s  n={c.n}{het_lbl}")

print(f"\n  Omnibus: H={h03_ttm.omnibus_statistic:.4f}, p={h03_ttm.omnibus_p:.6f}, sig={h03_ttm.omnibus_significant}")
if h03_ttm.pairwise:
    for pw in h03_ttm.pairwise:
        sig = "✗" if pw.significant else "✓"
        print(f"  {pw.pair:50s}: A12={pw.a12:.3f} ({pw.effect_magnitude}) p_adj={pw.p_value_adjusted:.4f} [{sig}]")


H-03: time_to_mitigate  |  significant_category_disparity
  application_fault   : pooled IQM=  310.6s  EW IQM=  313.4s  n=51
  network_fault       : pooled IQM=  437.2s  EW IQM=  439.7s  n=39
  resource_fault      : pooled IQM=  491.7s  EW IQM=  495.4s  n=70 ⚠️

  Omnibus: H=40.3417, p=0.000000, sig=True
  application_fault vs network_fault                : A12=0.350 (medium) p_adj=0.0305 [✗]
  application_fault vs resource_fault               : A12=0.124 (large) p_adj=0.0000 [✗]
  network_fault vs resource_fault                   : A12=0.430 (small) p_adj=0.2310 [✓]


---
## Step 4: Run H-03 — reasoning_quality_score


In [5]:
rsn_data = {}
for cat, runs in categories.items():
    rsn_data[cat] = build_subfault_data(runs, "reasoning_quality_score",
                                        detected_only=False, source="qualitative")

h03_rsn = run_cross_category_test(rsn_data, metric_name="reasoning_quality_score")

print(f"H-03: {h03_rsn.metric_name}  |  {h03_rsn.overall_assessment}")
print("=" * 85)
for c in h03_rsn.per_category:
    het_lbl = " ⚠️" if c.within_heterogeneous else ""
    print(f"  {c.category:20s}: pooled IQM={c.pooled_iqm:.2f}/10  EW IQM={c.equal_weight_iqm:.2f}/10  n={c.n}{het_lbl}")

print(f"\n  Omnibus: H={h03_rsn.omnibus_statistic:.4f}, p={h03_rsn.omnibus_p:.6f}, sig={h03_rsn.omnibus_significant}")
if h03_rsn.pairwise:
    for pw in h03_rsn.pairwise:
        sig = "✗" if pw.significant else "✓"
        print(f"  {pw.pair:50s}: A12={pw.a12:.3f} ({pw.effect_magnitude}) p_adj={pw.p_value_adjusted:.4f} [{sig}]")


H-03: reasoning_quality_score  |  significant_category_disparity
  application_fault   : pooled IQM=8.38/10  EW IQM=8.28/10  n=60
  network_fault       : pooled IQM=4.47/10  EW IQM=4.53/10  n=90
  resource_fault      : pooled IQM=6.68/10  EW IQM=6.60/10  n=90

  Omnibus: H=92.4389, p=0.000000, sig=True
  application_fault vs network_fault                : A12=0.891 (large) p_adj=0.0000 [✗]
  application_fault vs resource_fault               : A12=0.771 (large) p_adj=0.0000 [✗]
  network_fault vs resource_fault                   : A12=0.207 (large) p_adj=0.0000 [✗]


---
## Step 5: Run H-03 — hallucination_score


In [6]:
hal_data = {}
for cat, runs in categories.items():
    hal_data[cat] = build_subfault_data(runs, "hallucination_score",
                                        detected_only=False, source="qualitative")

h03_hal = run_cross_category_test(hal_data, metric_name="hallucination_score")

print(f"H-03: {h03_hal.metric_name}  |  {h03_hal.overall_assessment}")
print("=" * 85)
for c in h03_hal.per_category:
    het_lbl = " ⚠️" if c.within_heterogeneous else ""
    print(f"  {c.category:20s}: pooled IQM={c.pooled_iqm:.3f}  EW IQM={c.equal_weight_iqm:.3f}  n={c.n}{het_lbl}")

print(f"\n  Omnibus: H={h03_hal.omnibus_statistic:.4f}, p={h03_hal.omnibus_p:.6f}, sig={h03_hal.omnibus_significant}")
if h03_hal.pairwise:
    for pw in h03_hal.pairwise:
        sig = "✗" if pw.significant else "✓"
        print(f"  {pw.pair:50s}: A12={pw.a12:.3f} ({pw.effect_magnitude}) p_adj={pw.p_value_adjusted:.4f} [{sig}]")


H-03: hallucination_score  |  significant_category_disparity
  application_fault   : pooled IQM=0.060  EW IQM=0.060  n=60
  network_fault       : pooled IQM=0.230  EW IQM=0.230  n=90
  resource_fault      : pooled IQM=0.100  EW IQM=0.110  n=90

  Omnibus: H=59.7500, p=0.000000, sig=True
  application_fault vs network_fault                : A12=0.161 (large) p_adj=0.0000 [✗]
  application_fault vs resource_fault               : A12=0.343 (medium) p_adj=0.0011 [✗]
  network_fault vs resource_fault                   : A12=0.736 (large) p_adj=0.0000 [✗]


---
## Step 6: Summary & Interpretation


In [7]:
print("H-03 Cross-Category Comparison — Summary")
print("=" * 85)

results = {
    "time_to_detect": h03_ttd,
    "time_to_mitigate": h03_ttm,
    "reasoning_quality_score": h03_rsn,
    "hallucination_score": h03_hal,
}

for metric, r in results.items():
    sig_lbl = "❌ DISPARITY" if r.omnibus_significant else "✅ UNIFORM"
    print(f"\n  {metric}: {sig_lbl}  (KW H={r.omnibus_statistic:.2f}, p={r.omnibus_p:.4f})")
    if r.pairwise:
        sig_pairs = [p for p in r.pairwise if p.significant]
        if sig_pairs:
            print(f"    Significant pairs:")
            for p in sig_pairs:
                print(f"      {p.pair}: A12={p.a12:.3f} ({p.effect_magnitude})")
    # Flag heterogeneous categories
    het_cats = [c.category for c in r.per_category if c.within_heterogeneous]
    if het_cats:
        print(f"    ⚠️ Within-category heterogeneity in: {', '.join(het_cats)}")

print("\n" + "=" * 85)
print("\nInterpretation:")
print("  - DISPARITY: The agent performs significantly differently across fault categories.")
print("    Weakest category should be flagged in the certification report.")
print("  - UNIFORM: No evidence of performance difference across categories.")
print("  - ⚠️ Heterogeneous categories: sub-faults within the category behave differently.")
print("    Pooled comparison should be interpreted with caution for those categories.")


H-03 Cross-Category Comparison — Summary

  time_to_detect: ❌ DISPARITY  (KW H=55.16, p=0.0000)
    Significant pairs:
      application_fault vs network_fault: A12=0.626 (small)
      application_fault vs resource_fault: A12=0.059 (large)
      network_fault vs resource_fault: A12=0.287 (large)
    ⚠️ Within-category heterogeneity in: resource_fault

  time_to_mitigate: ❌ DISPARITY  (KW H=40.34, p=0.0000)
    Significant pairs:
      application_fault vs network_fault: A12=0.350 (medium)
      application_fault vs resource_fault: A12=0.124 (large)
    ⚠️ Within-category heterogeneity in: resource_fault

  reasoning_quality_score: ❌ DISPARITY  (KW H=92.44, p=0.0000)
    Significant pairs:
      application_fault vs network_fault: A12=0.891 (large)
      application_fault vs resource_fault: A12=0.771 (large)
      network_fault vs resource_fault: A12=0.207 (large)

  hallucination_score: ❌ DISPARITY  (KW H=59.75, p=0.0000)
    Significant pairs:
      application_fault vs network_fault:

---
## Step 7: JSON Output


In [8]:
output = {
    "hypothesis_id": "H-03",
    "hypothesis_name": "Cross-Category Performance Comparison",
    "null_hypothesis": "Performance is the same across all fault categories.",
    "alt_hypothesis": "At least one fault category has significantly different performance.",
    "certification_rule": "If disparity is significant with medium/large effect size, the weakest category is flagged.",
    "hypothesis_metadata": {
        "pipeline": "Shapiro-Wilk (informational) → Kruskal-Wallis (omnibus) → Mann-Whitney U + A12 (Holm-corrected)",
        "aggregation": "pooled per-run comparison across categories",
        "correction_method": "holm_bonferroni",
        "alpha": 0.05,
        "categories": list(categories.keys()),
        "n_runs_per_category": {cat: len(runs) for cat, runs in categories.items()},
        "mode": "both (always active)",
    },
    "results": {},
}

for metric, r in results.items():
    output["results"][metric] = {
        "overall_assessment": r.overall_assessment,
        "omnibus": {
            "test": r.test_used,
            "statistic": r.omnibus_statistic,
            "p_value": r.omnibus_p,
            "significant": r.omnibus_significant,
        },
        "per_category": [
            {
                "category": c.category,
                "n": c.n,
                "pooled_iqm": c.pooled_iqm,
                "equal_weight_iqm": c.equal_weight_iqm,
                "is_normal": c.is_normal,
                "within_heterogeneous": c.within_heterogeneous,
                "within_kw_p": c.within_kw_p,
            }
            for c in r.per_category
        ],
        "pairwise": [
            {
                "pair": p.pair,
                "p_value_adjusted": p.p_value_adjusted,
                "significant": p.significant,
                "a12": p.a12,
                "effect_magnitude": p.effect_magnitude,
            }
            for p in r.pairwise
        ],
    }

print(json.dumps(output, indent=2))


{
  "hypothesis_id": "H-03",
  "hypothesis_name": "Cross-Category Performance Comparison",
  "null_hypothesis": "Performance is the same across all fault categories.",
  "alt_hypothesis": "At least one fault category has significantly different performance.",
  "certification_rule": "If disparity is significant with medium/large effect size, the weakest category is flagged.",
  "hypothesis_metadata": {
    "pipeline": "Shapiro-Wilk (informational) \u2192 Kruskal-Wallis (omnibus) \u2192 Mann-Whitney U + A12 (Holm-corrected)",
    "aggregation": "pooled per-run comparison across categories",
    "correction_method": "holm_bonferroni",
    "alpha": 0.05,
    "categories": [
      "application_fault",
      "network_fault",
      "resource_fault"
    ],
    "n_runs_per_category": {
      "application_fault": 60,
      "network_fault": 90,
      "resource_fault": 90
    },
    "mode": "both (always active)"
  },
  "results": {
    "time_to_detect": {
      "overall_assessment": "significant